# Few-shot 與推理:2026 怎麼處理「想清楚」

## 模組脈絡:意圖收斂下,推理還需不需要手動引導?

2023–24 的 prompt 工程花很多力氣「教模型推理」——「Let's think step by step」、自我一致性、思維樹…。但在 gpt-5+ 時代,這些大多**過時甚至有害**。本筆記釐清三件事:

1. **few-shot 不再提升推理**,它在 2026 的價值只剩 **格式 / 風格對齊**(仍是高 ROI)。
2. **推理模型已內化 CoT**:顯式「一步一步思考」對它冗餘,甚至干擾其原生推理。
3. **推理深度交給 API**:用 `reasoning_effort`(minimal/low/medium/high)與 `verbosity` 控制,而不是 prompt 咒語。➡️ 三家 provider 的推理參數對照,見 **[M1] `01-uncontrollability/03-reasoning-thinking.ipynb`**(本章不重做)。

> 本章談「意圖層」要不要手動引導推理;模型「內部」如何推理與其不可見性,屬 M1。

## 0. 環境設定

In [ ]:
import os
from openai import OpenAI
client = OpenAI()  # 讀取 OPENAI_API_KEY
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

def get_completion(messages, model=OPENAI_MODEL, temperature=0, max_tokens=2000):
    try:
        resp = client.responses.create(
            model=model, input=messages, temperature=temperature, max_output_tokens=max_tokens,
        )
        return resp.output_text
    except Exception as e:
        return f"[API 錯誤] {type(e).__name__}: {e}"

## 1. Few-shot 的真正用途:格式 / 風格對齊

2026 的 few-shot **不是用來讓模型更會推理**,而是用幾個範例把「輸出長什麼樣」釘死——特定格式、特定語氣、難以用文字描述的 schema。下面用 2 個範例對齊「城市 ▸ emoji 天氣 | 建議」這個格式。

In [ ]:
messages = [
    {"role": "system", "content": "依範例的格式與風格輸出,只回一行。"},
    {"role": "user", "content": "台北"},
    {"role": "assistant", "content": "台北 ☔️ 悶熱有雨 | 建議帶傘"},
    {"role": "user", "content": "高雄"},
    {"role": "assistant", "content": "高雄 ☀️ 晴朗炎熱 | 防曬補水"},
    {"role": "user", "content": "台中"},
]
print(get_completion(messages, temperature=0.3))
# few-shot 讓「城市 emoji 天氣 | 建議」格式被精準對齊——這是 2026 few-shot 的主要價值。

### Few-shot 要點

- **多樣性(diversity)**:範例要涵蓋不同情況,避免模型過擬合單一樣態。
- **用 messages 對話形式**呈現範例(把示範「塞進 assistant 嘴裡」),比塞在一段長 user 文字更清楚。
- **格式對齊 ≠ 推理增強**:難題的正確率主要靠模型本身能力與 `reasoning_effort`,不是靠多給範例。
- 若要**強制**結構(而非靠範例引導),用 M3 的 structured output(`text_format` / schema)。

## 2. 內心 OS:用 XML 標籤分開「過程」與「答案」

有時你想讓模型先展開思路、但只把**最終答案**給使用者。可用 XML 標籤分隔,方便程式擷取 `<answer>`。

> 注意適用範圍:對 **gpt-5+ 推理模型通常不需要**(它內部已推理,且原始思路不可見);這招主要用在**非推理 / 輕量模型**(如 mini / flash 處理稍難任務),或你需要一段**可審計的中間步驟**時。

In [ ]:
user_message = (
    "假設 x = 100,依序計算:加 1、加 10、減 1、乘 2、減 20。\n"
    "請把計算過程放進 <thinking></thinking>,最後的 x 放進 <answer></answer>。"
)
resp = get_completion([{"role": "user", "content": user_message}], temperature=0)
print(resp)

import re
m = re.search(r"<answer>(.*?)</answer>", resp, re.S)
print("擷取答案:", m.group(1).strip() if m else "(未找到)")

## 3. 何時仍需手動引導推理?(決策框)

| 情境 | 2026 做法 |
|------|-----------|
| 用**推理模型**(gpt-5、o 系列、Claude/Gemini thinking) | 調 `reasoning_effort`/`verbosity`(API),**不要**寫「一步一步思考」 |
| 用**非推理 / 輕量模型**碰稍難任務 | 可給結構化步驟或 few-shot 格式錨;或直接升級模型 |
| 需要**可審計**的中間步驟 | 用內心 OS / XML 標籤分隔 |
| 想要**穩定的輸出結構** | 用 few-shot 對齊格式,或 M3 structured output 強制 |

> 心法:**給任務與約束,然後讓開**(give the task and constraints, then get out of the way)。推理深度是 API 旋鈕,不是 prompt 咒語。

## 4. 2026 已淘汰的「咒語型」技巧(課程不再教)

以下 2023–24 流行技巧,對 gpt-5+ 推理模型**冗餘甚至有害**,本課程已移除(僅列此供辨識):

| 舊技巧 | 為何 2026 過時 |
|--------|----------------|
| 「Let's think step by step」當推薦咒語 | 推理模型已內化 CoT;顯式要求反而可能干擾 |
| 「Take a deep breath…」 | 咒語式 prompt,對新模型無穩定效益 |
| 詳盡 5-why 連鎖 | 手刻推理鏈;改用 `reasoning_effort` 或工作流分解(見 03) |
| Self-consistency(多取樣多數決) | 成本高;推理模型單次品質已足,需要時用 M7 評估 |
| Tree-of-Thought(多專家分支) | 不穩定、昂貴;已被推理模型取代 |

> 共同教訓:**把推理交給模型與 API,把心力放在意圖收斂(spec)與驗證(M7)。**

---

## 本章小結

1. **few-shot = 格式 / 風格對齊**,不再是推理增強器;要強制結構用 M3 structured output。
2. **推理模型已內化 CoT**:別寫「一步一步思考」;推理深度用 `reasoning_effort`/`verbosity` 控制(細節見 M1)。
3. **內心 OS / XML** 仍有用:隱藏中間步驟、可審計、或非推理模型。
4. **淘汰咒語**:take a deep breath、5-why、self-consistency、ToT——交給模型與評估(M7)。
5. 心法:**給任務與約束,然後讓開**;把心力移到 spec(04)與驗證(M7)。